## Imports and Config

In [ ]:
%load_ext autoreload
%autoreload 2

Custom imports from the `config/settings.py` file.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT / "src"))

from sem_rag.config.settings import *

In [ ]:
import base64
import io
from PIL import Image
from tqdm import tqdm
import numpy as np
import torch

from openai import OpenAI
from openai.types.create_embedding_response import CreateEmbeddingResponse

In [ ]:
a = np.load(SEMDataset.CROPS / "38_a6a840c41ec3a131a5d6de121831afb7797e181926bfdfed50b62b3335899f52_512.npy")
print(a.dtype, a.shape, a.min(), a.max())

In [ ]:
client = OpenAI(base_url=VLLM.API_URL, api_key=VLLM.API_KEY)
models = client.models.list()
model = next(m for m in models.data if m.id == "Qwen/Qwen3-VL-Embedding-8B")
models.data

In [ ]:
INSTRUCTION = "Represent the user's input."
CROP_PATH = Path(SEMDataset.CROPS / "38_a6a840c41ec3a131a5d6de121831afb7797e181926bfdfed50b62b3335899f52_512.npy")

In [ ]:
def npy_crop_to_png_bytes(path: Path) -> bytes:
    arr = np.load(path)
    assert arr.dtype == np.uint8, f"{path}: expected uint8, got {arr.dtype}"
    assert arr.shape == (512, 512), f"{path}: expected (512, 512), got {arr.shape}"

    image = Image.fromarray(arr, mode="L")
    buf = io.BytesIO()
    image.save(buf, format="PNG")
    return buf.getvalue()

In [ ]:
def encode_image_data_url(path: Path) -> str:
    png_bytes = npy_crop_to_png_bytes(path)
    b64 = base64.b64encode(png_bytes).decode("utf-8")
    return f"data:image/png;base64,{b64}"

In [ ]:
api_key = VLLM.API_KEY
if not api_key:
    raise SystemExit(f"Set {VLLM.API_KEY} in the environment before running.")

client = OpenAI(base_url=VLLM.API_URL, api_key=api_key)
models = client.models.list()
model = next(m for m in models.data if m.id == VLLM.EMBEDDING_MODEL).id
print(f"Using model: {model}")

messages = [
    {"role": "system", "content": [{"type": "text", "text": INSTRUCTION}]},
    {"role": "user", "content": [
        {"type": "image_url", "image_url": {"url": encode_image_data_url(CROP_PATH)}},
        {"type": "text", "text": ""},
    ]},
    {"role": "assistant", "content": [{"type": "text", "text": ""}]},
]

response: CreateEmbeddingResponse = client.post(
    "/embeddings",
    cast_to=CreateEmbeddingResponse,
    body={
        "messages": messages,
        "model": model,
        "encoding_format": "float",
        "continue_final_message": True,
        "add_special_tokens": True,
    },
)
embedding = response.data[0].embedding
print(f"Embedding dim: {len(embedding)}")
print(f"First 10 values: {embedding[:10]}")

# Embedding

Embed SEM image crops with Qwen/Qwen3-VL-Embedding-8B via the deployed vLLM server.

Embed every crop in CROPS with Qwen3-VL-Embedding, saving one .pt file per
crop in EMBEDDINGS (same layout as your DINO embeddings). No async, no
merged checkpoint file -- if a crop's .pt file already exists, it's skipped,
so re-running after a crash just picks up where it left off for free.

In [ ]:
# Use the SAME instruction string you use when embedding text queries at retrieval
# time. Qwen3-VL-Embedding's own reference code applies one instruction symmetrically
# to both queries and documents (unlike the older text-only Qwen3-Embedding models,
# which recommend no instruction on the document side) -- so this needs to match your
# retrieval-side query embedding call, not just be "reasonable".
INSTRUCTION = "Represent the user's input."

In [ ]:
def crop_to_data_url(path: Path) -> str:
    arr = np.load(path)
    image = Image.fromarray(arr, mode="L")
    buf = io.BytesIO()
    image.save(buf, format="PNG")
    b64 = base64.b64encode(buf.getvalue()).decode("utf-8")
    return f"data:image/png;base64,{b64}"

In [ ]:
def embed_image(client: OpenAI, model: str, path: Path) -> list[float]:
    messages = [
        {"role": "system", "content": [{"type": "text", "text": INSTRUCTION}]},
        {"role": "user", "content": [
            {"type": "image_url", "image_url": {"url": crop_to_data_url(path)}},
            {"type": "text", "text": ""},
        ]},
        {"role": "assistant", "content": [{"type": "text", "text": ""}]},
    ]
    response: CreateEmbeddingResponse = client.post(
        "/embeddings",
        cast_to=CreateEmbeddingResponse,
        body={
            "messages": messages,
            "model": model,
            "encoding_format": "float",
            "continue_final_message": True,
            "add_special_tokens": True,
        },
    )
    return response.data[0].embedding

In [ ]:
SEMDataset.EMBEDDINGS_QWEN.mkdir(exist_ok=True)

client = OpenAI(base_url=VLLM.API_URL, api_key=VLLM.API_KEY)
models = client.models.list()
model = next(m for m in models.data if m.id == VLLM.EMBEDDING_MODEL).id
print(f"Using model: {model}")

crop_paths = sorted(SEMDataset.CROPS.glob("*.npy"))
for crop_path in tqdm(crop_paths):
    image_id = crop_path.stem.split("_")[0]
    out_path = SEMDataset.EMBEDDINGS_QWEN / f"{image_id}.pt"
    if out_path.exists():
        continue
    embedding = embed_image(client, model, crop_path)
    torch.save(torch.tensor(embedding), out_path)

print(f"Done: {len(crop_paths)} crops -> {SEMDataset.EMBEDDINGS_QWEN}/")